In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import os
from pathlib import Path

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/data"))

In [ ]:
# ── Group-aware split: prevents augmentation leakage ──
# A video's augmented copies must NEVER straddle train/test, or the test set
# is just near-duplicates of training data → fake-high accuracy.
# Fix: split ORIGINAL (processed) videos first, augment ONLY the train side,
# test ONLY on untouched originals.
import numpy as np
import os
from sklearn.model_selection import train_test_split

BASE = "/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/data"

TARGET_SIGNS = [
    "คนหูหนวก", "คุณ", "ช่วย", "ขอบคุณ",
    "ฉัน", "ต้องการ", "เข้าใจ", "ไม่", "ถาม", "บอก", "finish", "none"
]


# 1) Gather original processed videos as (path, label, sign, stem)
originals = []
for label_idx, sign in enumerate(TARGET_SIGNS):
    d = os.path.join(BASE, "processed", sign)
    if not os.path.exists(d):
        continue
    for f in os.listdir(d):
        if f.endswith(".npy"):
            originals.append((os.path.join(d, f), label_idx, sign, f[:-4]))

labels = [o[1] for o in originals]

# 2) Split the ORIGINALS (stratified so every sign appears in test)
train_orig, test_orig = train_test_split(
    originals, test_size=0.2, random_state=42, stratify=labels
)

# 3) TEST set = originals only (never augmented, never seen in training)
X_test = np.array([np.load(o[0]) for o in test_orig])
y_test = np.array([o[1] for o in test_orig])

# 4) TRAIN set = train originals + their augmented copies
train_keys = {(o[2], o[3]) for o in train_orig}   # (sign, stem) allowed in train
X_train, y_train = [], []
for o in train_orig:                               # the originals
    X_train.append(np.load(o[0])); y_train.append(o[1])
for label_idx, sign in enumerate(TARGET_SIGNS):    # their augmentations
    d = os.path.join(BASE, "augmented", sign)
    if not os.path.exists(d):
        continue
    for f in os.listdir(d):
        if not f.endswith(".npy"):
            continue
        orig_stem = f[:-4].rsplit("_aug_", 1)[0]   # aug name = {stem}_aug_{i}
        if (sign, orig_stem) in train_keys:
            X_train.append(np.load(os.path.join(d, f))); y_train.append(label_idx)

X_train = np.array(X_train); y_train = np.array(y_train)

print(f"Train: {X_train.shape}  (originals + their augments)")
print(f"Test:  {X_test.shape}  (untouched originals only)")
print(f"Leakage check: no video's augments cross the split ✓")

In [ ]:
# Split already done above (group-aware). Sanity check class balance:
import numpy as np
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {TARGET_SIGNS[u]:<12} test samples: {c}")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(32, return_sequences=True, input_shape=(30, 126)),
    Dropout(0.5),
    LSTM(32),
    Dropout(0.5),
    Dense(16, activation='relu'),
    Dense(12, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")
print(f"Stopped at epoch: {len(history.history['val_accuracy'])}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(13, 11))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=TARGET_SIGNS,
    yticklabels=TARGET_SIGNS,
    linewidths=0.5,
    linecolor='gray'
)
plt.title('Confusion Matrix — ThSL Bridge v6c (12 classes)',
          fontsize=14, pad=20)
plt.ylabel('Actual (ท่าจริง)', fontsize=12)
plt.xlabel('Predicted (ท่าที่ทำนาย)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/confusion_matrix_v6c.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

BASE = "/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/data"

def load_class_samples(sign_name):
    samples = []
    for folder in ["processed", "augmented"]:
        sign_dir = os.path.join(BASE, folder, sign_name)
        if not os.path.exists(sign_dir):
            continue
        for f in os.listdir(sign_dir):
            if f.endswith(".npy"):
                samples.append(np.load(os.path.join(sign_dir, f)))
    return np.array(samples)  # shape: (N, 30, 126)

none_data = load_class_samples("none")
khun_data = load_class_samples("คุณ")

# Average keypoint position across all frames and samples
none_mean = none_data.mean(axis=(0, 1))   # shape: (126,)
khun_mean = khun_data.mean(axis=(0, 1))

# Plot first 42 values = right hand x,y coordinates (21 landmarks × 2)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, data, title, color in zip(
    axes,
    [none_mean[:42], khun_mean[:42]],
    ["None (resting)", "คุณ (pointing)"],
    ["steelblue", "tomato"]
):
    ax.bar(range(42), data, color=color, alpha=0.7)
    ax.set_title(f"Average Right Hand Keypoints — {title}", fontsize=12)
    ax.set_xlabel("Keypoint index (x0,y0,x1,y1...)")
    ax.set_ylabel("Normalized coordinate value")
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/none_vs_khun_overlap.png', dpi=150)
plt.show()

# Print distance between the two averages
distance = np.linalg.norm(none_mean - khun_mean)
print(f"Euclidean distance between None and คุณ mean vectors: {distance:.4f}")
print("< 5.0 = dangerously close, > 10.0 = well separated")

In [ ]:
y_pred_proba = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred_proba, axis=1)
y_confidence = np.max(y_pred_proba, axis=1)

# For each sign, what's the average confidence when correct vs wrong?
for i, sign in enumerate(TARGET_SIGNS):
    mask = y_test == i
    if mask.sum() == 0:
        continue
    correct = (y_pred_classes[mask] == y_test[mask])
    conf_correct = y_confidence[mask][correct].mean() if correct.sum() > 0 else 0
    conf_wrong = y_confidence[mask][~correct].mean() if (~correct).sum() > 0 else 0
    print(f"{sign}: correct_conf={conf_correct:.2f} | wrong_conf={conf_wrong:.2f} | n_correct={correct.sum()}/{mask.sum()}")

In [ ]:
model.save("/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/thsl_model_v6c.keras")
print("Model v6c saved!")

In [ ]:
model.save_weights("/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/thsl_model_v6c.weights.h5")
print("Weights v6c saved!")